# 04 - Feature Engineering

Computers can't do math on the word `"Married"` — every text column has to become a number. This module also splits the data into training (70%) and test (30%) sets **before** any further processing, so nothing from the test set leaks into training.

**Ordinal** columns (real order, e.g. `"No" < "Yes"`) are numbered 0, 1, 2...
**Nominal** columns (no real order, e.g. employment type) are left as text here — they get one-hot encoded inside the modelling pipeline in Module 5, so that the encoding is learned only from training data.

**Input:** `../03-data-preprocessing/data/cleaned_data.csv`
**Output:** `data/X_train.csv`, `data/X_test.csv`, `data/y_train.csv`, `data/y_test.csv`


In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = "../03-data-preprocessing/data/cleaned_data.csv"
TARGET = "Default"
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Text columns where the categories have a real ORDER (low to high)
ORDINAL_MAPS = {
    "Education":     ["High School", "Bachelor's", "Master's", "PhD"],
    "HasMortgage":   ["No", "Yes"],
    "HasDependents": ["No", "Yes"],
    "HasCoSigner":   ["No", "Yes"],
}

TEST_SIZE = 0.30
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")


Loaded 255,347 rows, 17 columns


In [2]:
qualitative = df.select_dtypes(exclude=[np.number]).columns.tolist()
ordinal_cols = [c for c in qualitative if c in ORDINAL_MAPS]
nominal_cols = [c for c in qualitative if c not in ORDINAL_MAPS]

print(f"Ordinal (real order) - {len(ordinal_cols)}:")
for c in ordinal_cols:
    print(f"   {c:<16} {' < '.join(ORDINAL_MAPS[c])}")

print(f"\nNominal (no real order) - {len(nominal_cols)}:")
for c in nominal_cols:
    print(f"   {c:<16} categories: {df[c].dropna().unique().tolist()}")

for col, order in ORDINAL_MAPS.items():
    mapping = {level: i for i, level in enumerate(order)}
    df[col] = df[col].map(mapping)

print(f"\nDone. {ordinal_cols} are now numbers.")
print(f"{nominal_cols} stay as text - Module 5's pipeline one-hot encodes them.")


Ordinal (real order) - 4:
   Education        High School < Bachelor's < Master's < PhD
   HasMortgage      No < Yes
   HasDependents    No < Yes
   HasCoSigner      No < Yes

Nominal (no real order) - 3:
   EmploymentType   categories: ['Full-time', 'Unemployed', 'Self-employed', 'Part-time']
   MaritalStatus    categories: ['Divorced', 'Married', 'Single']
   LoanPurpose      categories: ['Other', 'Auto', 'Business', 'Home', 'Education']

Done. ['Education', 'HasMortgage', 'HasDependents', 'HasCoSigner'] are now numbers.
['EmploymentType', 'MaritalStatus', 'LoanPurpose'] stay as text - Module 5's pipeline one-hot encodes them.


**The mistake to avoid:** numbering nominal categories like 0, 1, 2, 3 anyway — that secretly tells the model "Self-employed is more than Full-time", which is meaningless and would confuse it. That's why nominal columns are kept as text here and one-hot encoded later, inside the pipeline.

In [3]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training rows : {len(X_train):,}  ({(1-TEST_SIZE)*100:.0f}%)")
print(f"Test rows     : {len(X_test):,}  ({TEST_SIZE*100:.0f}%)")
print(f"\nDefault rate in training set : {y_train.mean()*100:.2f}%")
print(f"Default rate in test set     : {y_test.mean()*100:.2f}%")
print("(Both should be almost identical - that's the stratify option working.)")

X_train.to_csv(os.path.join(OUTPUT_DIR, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(OUTPUT_DIR, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(OUTPUT_DIR, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(OUTPUT_DIR, "y_test.csv"), index=False)
print("\nSaved train/test splits to data/")


Training rows : 178,742  (70%)
Test rows     : 76,605  (30%)

Default rate in training set : 11.61%
Default rate in test set     : 11.61%
(Both should be almost identical - that's the stratify option working.)



Saved train/test splits to data/


## How to read this

- We split **before** doing anything else, so no information from the "exam" (test set) sneaks into the "studying" (training set).
- `stratify=y` keeps the 11.6% default rate the same in both piles, since a random split could otherwise land unluckily given how rare defaults are.
- Both piles ended up at **11.61% default rate**, confirming the split worked.

## Next module
Module 5 (Model Training) builds a preprocessing pipeline (scaling + one-hot encoding) and trains 6 different models on `X_train`/`y_train`.
